# 77장 · 검토한 이미지 후보를 적용하고 복구 판단하기

배포 직전, 승인 화면에 적힌 이미지와 CI가 검사한 이미지의 이름은 같은데 digest가 다릅니다. 어떤 버전을 실제로 적용할지 정하지 않으면 되돌릴 때도 근거가 없습니다. 한 배포의 승인 조건과 실패 후 복구 경로를 로컬 기록으로 연습합니다.

실행을 확인한 커널은 Python 3.12.15입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 과제 셀에 직접 코드를 작성한 뒤 바로 아래에서 풀이를 펼쳐 보세요.

## 같은 태그 아래 서로 다른 후보 구별하기

CI는 검사한 소스와 이미지 후보를 함께 인계합니다. CD는 환경 승인과 현재 배포 상태를 확인하고 그 후보만 적용합니다. 여기서는 작은 바이트열의 SHA-256으로 서로 다른 내용을 구별합니다. 실제 레지스트리에서 받은 manifest digest를 대신하는 수업용 값입니다.



**볼 곳**　‘이미지 식별값’를 기준으로 양쪽의 ‘같은 태그’와 ‘후보 대조’를 대조하세요.

**애니메이션의 의미**　태그가 같아도 실제 이미지 식별값은 다를 수 있습니다.


### 셀 01 · 태그가 같아도 내용이 바뀐 경우

내용 한 줄이 바뀌었을 때 같은 승인값을 계속 쓸 수 있는지 확인하세요.

In [1]:
import hashlib
def digest(data):
    return "sha256:" + hashlib.sha256(data).hexdigest()
approved = digest(b"course-image B")
candidate = digest(b"course-image C")
print(approved == candidate)

False


False입니다. 승인 당시 검토한 내용과 현재 후보가 다르면 배포를 중단하고 새 후보의 검사 기록을 확인합니다.

### 과제 02 · 배포 직전 두 조건 확인

승인한 digest와 후보 digest가 같고, 사람이 해당 환경의 적용을 승인했을 때만 배포를 허용하세요.

In [2]:
# can_apply(expected, candidate, approved_by_reviewer)를 작성하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>def can_apply(expected, candidate, approved_by_reviewer):
    return expected == candidate and approved_by_reviewer

print(can_apply(approved, candidate, True))</code></pre>
<p>실행 결과</p><pre><code>False</code></pre>
<p>사람이 승인했다고 표시해도 digest가 달라 False입니다. 수동 실행 버튼, 환경 승인, 후보 동일성은 각각 따로 확인합니다.</p>
</details>

## 적용 뒤 기능 검사가 실패한 경우

아래 배포 대상은 수업용 메모리 사전입니다. 검사된 후보를 적용한 뒤 과제 안내 질문이 근거 ID를 유지하는지 확인합니다. 실패하면 보관한 이전 이미지로 돌립니다. DB 내용은 이 작업에서 바뀌지 않습니다.



**볼 곳**　‘후보 적용’에서 시작해 ‘기능 검사 실패’를 거친 뒤 ‘이전 버전 복귀’에서 다시 판단하는 흐름을 따라가세요.

**애니메이션의 의미**　기능 확인이 실패하면 검증한 이전 버전으로 돌아갑니다.


### 셀 03 · 근거가 없는 후보 되돌리기

health가 정상이더라도 문서 근거가 사라진 후보를 어떻게 처리하는지 보세요.

앞 과제에서는 승인이 가리키는 후보와 적용할 후보가 달라서 멈췄습니다. 이번에는 B의 승인이 유효해 적용했고, 그 뒤 검사가 실패한 상황입니다. 되돌리기를 판단하는 시점이 다릅니다.

In [3]:
versions = {"A": {"health": 200, "source": "course-guide-v1:p2"},
            "B": {"health": 200, "source": None}}
deployment = {"active": "A"}
previous = deployment["active"]
deployment["active"] = "B"
current = versions[deployment["active"]]
if current["health"] != 200 or current["source"] != "course-guide-v1:p2":
    deployment["active"] = previous
print(deployment["active"], versions[deployment["active"]]["source"])

A course-guide-v1:p2


A와 원래 근거 ID가 보입니다. 이 결과는 복구 조건을 로컬에서 재현한 것입니다. 실제 배포에서는 이전 이미지와 현재 스키마가 호환되는지, 이전 버전에서도 같은 문서가 조회되는지 별도 검증해야 합니다.

## 돌아보기

운영 인계에는 적용한 후보 digest, 이전 digest, 확인한 요청과 결과, 적용하거나 되돌린 시각을 남깁니다. 상태를 읽을 수 없을 때 임의로 배포를 반복하지 않도록 담당자를 정하세요.

### 후보에서 확인 완료까지



**볼 곳**　후보의 digest와 배포 뒤 출처가 어느 검증 단계에서 비교되는지 보세요.

**애니메이션의 의미**　검토한 이미지와 적용할 이미지가 같아야 하고, 적용 뒤에는 서비스 결과까지 다시 확인합니다.
